In [24]:
!pip install selenium webdriver-manager beautifulsoup4

In [34]:
!pip install selenium webdriver-manager beautifulsoup4

import time
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from webdriver_manager.chrome import ChromeDriverManager


def extract_stock_data(html):
    soup = BeautifulSoup(html, "html.parser")

    date_element = soup.select_one(".ChartHeader_date__fy3_Y")
    value_elements = soup.select(".ChartHeader_value__thRnp")

    if date_element is None or len(value_elements) < 4:
        return None

    date = date_element.text.replace("/", "-")

    open_price = value_elements[0].text
    high_price = value_elements[1].text
    low_price = value_elements[2].text
    close_price = value_elements[3].text

    return [
        date,
        open_price,
        high_price,
        low_price,
        close_price
    ]


def get_stock_values(driver, url):
    driver.get(url)
    time.sleep(8)

    # iframeを取得
    iframes = driver.find_elements(By.TAG_NAME, "iframe")

    # グラフがあるiframe番号
    target_iframe_index = 3

    driver.switch_to.frame(iframes[target_iframe_index])

    # Highchartsのグラフ要素を取得
    graph = driver.find_element(By.CSS_SELECTOR, "[data-highcharts-chart]")

    width = graph.size["width"]
    height = graph.size["height"]

    actions = ActionChains(driver)

    # グラフの中央へ移動
    actions.move_to_element_with_offset(
        graph,
        width // 2,
        height // 2
    ).perform()

    time.sleep(1)

    # グラフ右端へ移動
    actions.move_by_offset(
        width // 2 - 50,
        0
    ).perform()

    time.sleep(1)

    stock_data_list = []
    previous_data = None

    # 1pxずつ左へ移動
    for _ in range(int(width)):
        html = driver.page_source

        stock_data = extract_stock_data(html)

        if stock_data is not None and stock_data != previous_data:
            stock_data_list.append(stock_data)
            previous_data = stock_data

        ActionChains(driver).move_by_offset(-1, 0).perform()

    driver.switch_to.default_content()

    return stock_data_list


start_time = time.time()

options = webdriver.ChromeOptions()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")

service = Service(ChromeDriverManager().install())

driver = webdriver.Chrome(
    service=service,
    options=options
)

url = "https://www.nikkei.com/markets/worldidx/chart/nk225/?type=6month"

stock_values = get_stock_values(driver, url)

driver.quit()

end_time = time.time()

print("日付, 始値, 高値, 安値, 終値")

for stock in stock_values:
    print(stock)

print(f"処理時間: {end_time - start_time:.2f}秒")

日付, 始値, 高値, 安値, 終値
['2026-06-12', '65176.23', '67065.94', '64998.11', '66020.04']
処理時間: 231.82秒
